# Huấn luyện AI Core trên Kaggle GPU (T4 free)

Train 6 mô hình tự code (from-scratch, không pretrained) của dự án **Phân tích Video & Highlight AI**.

## Chuẩn bị (bắt buộc)
1. **Settings → Accelerator → GPU T4 x2**. KHÔNG chọn P100: P100 là sm_60, torch của Kaggle chỉ
   build cho sm_70+ → `no kernel image is available`. Cell 0 có guard tự fallback CPU, nhưng CPU
   chậm hơn nhiều (6 lần train Two-Tower ~30 phút).
2. **Settings → Internet → ON** (để tải MovieLens + corpus).
3. Đưa `ai-core` vào Kaggle: `bash dist/kaggle/push_kaggle.sh <username>` (đẩy cả dataset + notebook),
   hoặc **Add Data → Upload Dataset** đặt tên `ai-core`.

Notebook chạy **Two-Tower end-to-end** ngay (MovieLens tải tự động) → ra `best.pt` thật + metric.
Các model cần dataset lớn (ViT/ASR/Mini-GPT) có **cell template** — attach dataset rồi chạy.


## 0. Kiểm tra GPU + nạp ai_core

In [ ]:
import torch, sys, glob, os, zipfile
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Guard GPU tương thích: P100 (sm_60) không chạy được torch mới của Kaggle (cần sm_70+ = T4).
# Thử 1 phép cuda nhỏ; lỗi 'no kernel image' → fallback CPU để không crash.
if torch.cuda.is_available():
    try:
        _ = (torch.zeros(1, device='cuda') + 1).cpu()
        print('GPU dùng được:', torch.cuda.get_device_name(0))
    except Exception as e:
        print('GPU KHÔNG tương thích (', e, ') -> fallback CPU. Chọn Accelerator = GPU T4 x2 để dùng GPU.')
        torch.cuda.is_available = lambda: False

# Nạp ai_core: dataset có thể là thư mục ai_core/ HOẶC ai_core.zip (tùy cách upload) → xử lý cả 2.
# CHỈ nhận dir ai_core CÓ __init__.py (tránh folder giải nén rỗng/namespace do zip lồng nhau).
dirs = [d for d in glob.glob('/kaggle/input/**/ai_core', recursive=True)
        if os.path.isfile(os.path.join(d, '__init__.py'))]
zips = glob.glob('/kaggle/input/**/ai_core.zip', recursive=True)
if dirs:
    sys.path.insert(0, os.path.dirname(dirs[0]))
    print('ai_core (dir) từ', os.path.dirname(dirs[0]))
elif zips:
    dst = '/kaggle/working/aicore_pkg'
    with zipfile.ZipFile(zips[0]) as z: z.extractall(dst)
    sys.path.insert(0, dst)
    print('ai_core giải nén từ', zips[0], '->', dst)
else:
    raise SystemExit('KHÔNG thấy ai_core trong /kaggle/input — Add Data dataset ai-core vào notebook')

import ai_core
print('ai_core loaded:', getattr(ai_core, '__file__', '?'), '| version', getattr(ai_core, '__version__', '?'))
from ai_core.primitives import RMSNorm   # xác nhận package đầy đủ (không phải namespace rỗng)
print('primitives OK')


## 1. Sanity check toán học (nhanh, xác nhận code + GPU)

In [ ]:
# Kiểm chứng khối nguyên thủy trước khi train thật
import torch, copy, math
from ai_core.primitives import MultiHeadAttention, RMSNorm, build_rope_cache
from ai_core.optim import AdamW, split_decay_params
from torch import nn

# SDPA parity
m = MultiHeadAttention(16, 4, causal=True, use_sdpa=False).eval()
m2 = copy.deepcopy(m); m2.use_sdpa = True
x = torch.randn(2, 7, 16)
print('SDPA parity :', torch.allclose(m(x), m2(x), atol=1e-5))

# AdamW parity vs torch
torch.manual_seed(0)
a = nn.Linear(8,1).double(); b = copy.deepcopy(a)
oa = AdamW(split_decay_params(a), lr=1e-2, weight_decay=0.1)
ob = torch.optim.AdamW(split_decay_params(b), lr=1e-2, betas=(0.9,0.95), weight_decay=0.1, eps=1e-8)
X = torch.randn(32,8,dtype=torch.float64); Y = torch.randn(32,1,dtype=torch.float64)
for _ in range(100):
    oa.zero_grad(); ((a(X)-Y)**2).mean().backward(); oa.step()
    ob.zero_grad(); ((b(X)-Y)**2).mean().backward(); ob.step()
print('AdamW parity:', max((p-q).abs().max().item() for p,q in zip(a.parameters(), b.parameters())) < 1e-6)


## 2. Two-Tower — train end-to-end trên MovieLens-100k (chạy ngay)

Dữ liệu người tạo, công khai (hợp lệ dataset_spec §0). Chứng minh full loop train → checkpoint → metric.


In [ ]:
# MovieLens: ưu tiên đọc từ dataset đã attach (khỏi cần Internet); fallback tải qua internet
import os, glob, zipfile, urllib.request
found = glob.glob('/kaggle/input/**/u.data', recursive=True)
if found:
    ML = os.path.dirname(found[0])
    print('MovieLens từ dataset (offline):', ML)
else:
    ML = '/kaggle/working/ml-100k'
    if not os.path.exists(ML):
        z = '/kaggle/working/ml.zip'
        urllib.request.urlretrieve('https://files.grouplens.org/datasets/movielens/ml-100k.zip', z)
        with zipfile.ZipFile(z) as f: f.extractall('/kaggle/working')
    print('MovieLens tải qua Internet:', ML)
print('files:', sorted(os.listdir(ML))[:6])


In [ ]:
# Parse u.item/u.data → split PER-USER leave-last-k: fit | val (chọn hyperparameter) | test (đóng băng)
import torch
from ai_core.training.datasets import build_features, leave_last_k_split

n_genre = 19
item_genres = {}
with open(f'{ML}/u.item', encoding='latin-1') as f:
    for line in f:
        p = line.rstrip('\n').split('|')
        item_genres[int(p[0])] = torch.tensor([float(x) for x in p[-n_genre:]], dtype=torch.float32)

rows = []
with open(f'{ML}/u.data') as f:
    for line in f:
        u, i, r, ts = map(int, line.split())
        if r >= 4: rows.append((u, i, ts))

# leave_last_k thay time_split GLOBAL. Global cut để lại đúng 91/943 user có test positive (chỉ
# user hoạt động cuối kỳ), và làm baseline popularity mạnh lên hẳn ở test so với val — đủ để đảo
# chiều kết luận T1.7. Per-user vẫn không train trên tương lai của chính user đó, nhưng giữ đủ user.
K = 10                                    # 10 held-out/user → Recall@10 có thể đạt 1.0
train_rows, test_rows = leave_last_k_split(rows, k=K)
fit_rows, val_rows = leave_last_k_split(train_rows, k=K)

# Feature build từ fit_rows: mọi user đều có mặt trong fit (tính chất của leave-last-k) nên không
# ai bị rơi khỏi eval. log-popularity đếm từ fit thôi — đếm lẫn val/test là rò rỉ.
user_feat, video_feat = build_features(fit_rows, item_genres)
print('items', len(item_genres), '| pos fit', len(fit_rows), 'val', len(val_rows), 'test', len(test_rows))
print('user fit', len({u for u, _, _ in fit_rows}),
      '| val', len({u for u, _, _ in val_rows}),
      '| test', len({u for u, _, _ in test_rows}))
print('user_dim', next(iter(user_feat.values())).numel(),
      '| video_dim', next(iter(video_feat.values())).numel(), '(genre 19 + log-pop 1 + one-hot)')


In [ ]:
# Một hàm = train + early-stop theo val + eval, dùng cho CẢ sweep lẫn chốt test → không có đường
# nào để hai lần đo lệch cấu hình/tập user (đúng lớp bug đã gặp ở T1.7).
import torch
from torch.utils.data import DataLoader
from ai_core.config import ModelConfig, TrainConfig
from ai_core.models.two_tower import TwoTower
from ai_core.training import Trainer
from ai_core.training.checkpoint import load_checkpoint
from ai_core.training.datasets import BPRTripleDataset
from ai_core.training.loss_fns import two_tower_softmax_loss
from ai_core.scripts.evaluate import eval_recommender, format_recommender_report


def run_two_tower(eval_rows, *, temperature, seed=1337, max_steps=5000, lr=1e-3,
                  eval_every=500, ckpt_dir='/kaggle/working/ckpt_tmp'):
    """Train trên fit_rows, chọn checkpoint tốt nhất theo VAL, rồi chấm `eval_rows`.

    In-batch sampled softmax thay BPR-1-negative: batch 256 → 255 negative miễn phí. BPR chỉ
    1 negative/positive là nút thắt thật, không phải phân phối negative.
    neg_alpha=1.0 BẮT BUỘC: softmax cần log_q = log tần suất dữ liệu (xem two_tower_softmax_loss).

    Early-stop luôn theo VAL, kể cả khi eval_rows=test_rows — chọn checkpoint theo test là leak.
    Vì vậy train chỉ dùng fit_rows ở mọi lần gọi: `val` phải sạch để làm mốc chọn checkpoint.
    """
    ds = BPRTripleDataset(fit_rows, user_feat, video_feat, len(video_feat), seed=seed, neg_alpha=1.0)
    ld = DataLoader(ds, batch_size=256, shuffle=True, drop_last=True)
    user_dim = next(iter(user_feat.values())).numel()
    video_dim = next(iter(video_feat.values())).numel()
    mcfg = ModelConfig(name='two_tower', proj_dim=128, temperature=temperature,
                       user_dim=user_dim, video_dim=video_dim)
    model = TwoTower(mcfg, user_dim=user_dim, video_dim=video_dim)
    # loss + neg_alpha PHẢI nằm trong TrainConfig, dù ở đây chúng được truyền thẳng vào
    # Trainer(loss_fn=...) và BPRTripleDataset(neg_alpha=...). Nếu để trống thì train_config lưu
    # trong checkpoint ghi loss='' và neg_alpha=0.0 — sai so với thứ thật sự đã chạy, và người
    # đọc checkpoint sau này không tái lập được.
    tcfg = TrainConfig(model='two_tower', lr=lr, max_steps=max_steps, warmup_ratio=0.05,
                       batch_size=256, seed=seed, ckpt_dir=ckpt_dir, log_dir='',
                       eval_every=eval_every, metric_name='ndcg_at_10',
                       loss='two_tower_softmax', neg_alpha=1.0)

    # Trainer đưa `val_loader` thẳng vào compute_metric mà không tự iterate → truyền list row
    # val, để metric là NDCG@10 THẬT (full ranking) chứ không phải NDCG in-batch của
    # _two_tower_metric. Không có cách nào khác để early-stop theo đúng metric của cổng T1.7.
    def val_ndcg(m, rows, device):
        return eval_recommender(m, user_feat, video_feat, fit_rows, rows, k=10, device=device)['model_ndcg']

    trainer = Trainer(model, ld, val_rows, tcfg, loss_fn=two_tower_softmax_loss, model_cfg=mcfg,
                      compute_metric=val_ndcg)
    stats = trainer.train()

    # Model trong bộ nhớ là step CUỐI; nạp lại best.pt (chọn theo val) nếu không thì early-stop vô nghĩa.
    best = load_checkpoint(f'{ckpt_dir}/two_tower/best.pt', map_location=trainer.device)
    model.load_state_dict(best['state_dict'])
    m = eval_recommender(model, user_feat, video_feat, fit_rows, eval_rows, k=10, device=trainer.device)
    m['final_loss'] = float(stats['final_loss'])
    m['best_val_ndcg'] = float(best['metrics'].get('ndcg_at_10', float('nan')))
    m['best_step'] = int(best['metrics'].get('step', -1))
    return m, model, trainer


In [ ]:
# T1.7 đã chốt PASS và số liệu ghi ở docs/T1.7_bao_cao.md (v14: NDCG 0.0990±0.0026 vs pop 0.0760).
# Chạy lại tốn ~19 phút GPU mỗi lần push, và nếu cell sau lỗi thì papermill dừng cả notebook nên
# 19 phút đó mất trắng. Bật RUN_TWO_TOWER=True khi cần tái lập lại T1.7.
RUN_TWO_TOWER = False

if RUN_TWO_TOWER:
    # Sweep temperature TRÊN VAL — test vẫn đóng băng. temperature là knob duy nhất còn ý nghĩa:
    # sim ∈ [-1,1] sau L2-norm, còn log_q ≈ -7, temperature quyết định tỉ lệ giữa hai số đó.
    # neg_alpha cố định 1.0 (softmax cần đúng giá trị này), không sweep.
    sweep = {}
    for t in (0.05, 0.1, 0.2):
        m, _, _ = run_two_tower(val_rows, temperature=t, ckpt_dir=f'/kaggle/working/ckpt_val_t{t}')
        sweep[t] = m
        print(f'--- temperature={t} | VAL ---')
        print(format_recommender_report(m),
              f"| loss {m['final_loss']:.4f} | best_step {m['best_step']}")

    best_temp = max(sweep, key=lambda t: sweep[t]['model_ndcg'])
    print('\n=> chọn temperature =', best_temp, '(theo val NDCG@10, chưa hề nhìn test)')
else:
    best_temp = 0.1                 # giá trị val đã chọn ở v14
    print('BỎ QUA Two-Tower (RUN_TWO_TOWER=False) — dùng best_temp =', best_temp)


In [ ]:
# CHỐT T1.7: 3 seed, chấm TEST đúng một lần (evaluation_plan §0.2 + §0.3).
# Cổng = NDCG@10 vượt popularity (§1 cột 'Mục tiêu chấp nhận'). Recall@10 vẫn là metric CHÍNH
# phải báo cáo (metric phụ của Two-Tower là Coverage + thời gian truy vấn Milvus), chỉ không
# nằm trong điều kiện pass.
import shutil
from ai_core.scripts.evaluate import aggregate_runs, format_seed_report

if RUN_TWO_TOWER:
    runs = []
    for seed in (1337, 7, 2024):
        m, model, trainer = run_two_tower(test_rows, temperature=best_temp, seed=seed,
                                          ckpt_dir=f'/kaggle/working/ckpt_seed{seed}')
        runs.append(m)
        print(f'--- seed {seed} | TEST ---')
        print(format_recommender_report(m),
              f"| loss {m['final_loss']:.4f} | best_step {m['best_step']}"
              f" | val_ndcg@best {m['best_val_ndcg']:.4f}")
        if seed == 1337:
            # Checkpoint xuất = seed mặc định, KHÔNG phải seed tốt nhất trên test (chọn theo test = leak).
            shutil.rmtree('/kaggle/working/checkpoints', ignore_errors=True)
            shutil.copytree('/kaggle/working/ckpt_seed1337', '/kaggle/working/checkpoints')

    print()
    print(format_seed_report(aggregate_runs(runs)))
else:
    print('BỎ QUA chốt T1.7 (RUN_TWO_TOWER=False) — số liệu ở docs/T1.7_bao_cao.md §5.3')


## 2b. Siamese ViT — self-supervised trên keyframe SoccerNet (T1.3c)

Không cần nhãn tay: cặp dương là 2 keyframe **cùng hiệp cách 1–10s** + augment mạnh, negative là
in-batch (`dataset_spec §3.2`). Dữ liệu: 1 frame/5s trích từ 40 hiệp SoccerNet 224p, resize
224×224, JPEG q90 (~360 MB). Bản gốc PNG 1fps (111.066 frame, 11,8 GB) giữ ở máy local để trích
embedding — không upload.

Backbone này sau đó **đóng băng** làm nguồn token ảnh cho Highlight Fusion. `evaluation_plan §2.2`
cho phép Siamese đóng băng thay ViT phân loại, nên T1.5 **không phải chờ** ViT phân loại (T1.3a/b)
— đó là lý do làm được T1.5 mà chưa cần gom 300 video 10 danh mục.

**Hạn chế phải ghi vào luận án:** train toàn bộ trên bóng đá Premier League 2014–2016, cùng một
nhà sản xuất broadcast → backbone hẹp. Với highlight bóng đá là lợi thế; với visual search tổng
quát phải mở rộng nguồn (SoccerNet có `spain_laliga`, `germany_bundesliga`, `italy_serie-a`).


In [ ]:
# T1.3c đã chạy xong ở v22 (Recall@10 0.495, ghi trong docs/). Bật lại khi cần train lại —
# để True thì mỗi lần chạy kernel tốn thêm ~40 phút GPU mà không thêm thông tin gì.
RUN_SIAMESE = False
if RUN_SIAMESE:
    # Train Siamese ViT. Bốn cái bẫy đã cắn thật, giữ comment để không lặp:
    #  1. Kaggle TỰ GIẢI NÉN archive khi tạo dataset: upload .zip nhưng /kaggle/input nhận về thư mục
    #     giải nén sẵn, KHÔNG có .zip. Phải xử lý cả hai đường (giống guard nạp ai_core ở cell 0).
    #  2. Manifest sinh trên Windows nên path chứa '\'. os.path.basename ở Linux KHÔNG coi '\' là
    #     separator → parse ra cả đoạn đường dẫn. Nên dựng lại tên file từ group + sec, không parse path.
    #  3. /kaggle/input là filesystem MẠNG. Đọc 26k JPEG lẻ mỗi epoch từ đó rất chậm — v18 mất >1,5 giờ
    #     cho 8000 step. Phải copy sang đĩa local container một lần rồi mới train.
    #  4. NHƯNG /kaggle/working là thư mục OUTPUT: copy 26k ảnh vào đó thì Kaggle lưu hết làm output,
    #     và `kaggle kernels output` sau này tải lại đúng 372 MB ảnh đã có ở local (v20 bị đúng lỗi này,
    #     tải log mất >10 phút). Dùng /tmp — cũng là đĩa local nhưng KHÔNG vào output.
    import glob, json, os, shutil, time, zipfile, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import SiamesePairDataset
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts.train import _siamese_metric      # Recall@10 trên toàn bộ val, dùng lại thay vì viết lại

    JPG_DIR, MAN_NAME = 'siamese_shots_jpg', 'siamese_shots.json'
    LOCAL = '/tmp/siamese_local'                   # KHÔNG /kaggle/working: xem bẫy 4

    src_dirs = [d for d in glob.glob(f'/kaggle/input/**/{JPG_DIR}', recursive=True) if os.path.isdir(d)]
    if src_dirs:
        SRC = os.path.dirname(src_dirs[0])
        print(f'{JPG_DIR} (dir) từ', SRC)
    else:
        z = glob.glob('/kaggle/input/**/*.zip', recursive=True)
        if not z:
            raise SystemExit(f'Không thấy {JPG_DIR}/ lẫn .zip nào trong /kaggle/input')
        SRC = '/tmp/siamese_unzip'
        with zipfile.ZipFile(z[0]) as f:
            f.extractall(SRC)
        print('giải nén', z[0])

    if not os.path.isdir(os.path.join(LOCAL, JPG_DIR)):
        t = time.time()
        shutil.copytree(os.path.join(SRC, JPG_DIR), os.path.join(LOCAL, JPG_DIR))
        print(f'copy → đĩa local trong {time.time() - t:.0f}s')

    man_path = glob.glob(os.path.join(SRC, '**', MAN_NAME), recursive=True)
    if not man_path:
        raise SystemExit(f'Thiếu {MAN_NAME} trong {SRC}')
    man = json.load(open(man_path[0], encoding='utf-8'))
    for it in man['items']:
        fname = f"{it['group']}_s{it['extra']['sec']:05d}.jpg"     # dựng lại, không parse path Windows
        it['path'] = os.path.join(LOCAL, JPG_DIR, it['group'], fname)
        it['extra'].pop('src', None)                               # path PNG gốc chỉ có ở máy local
    MANIFEST = '/tmp/siamese_kaggle.json'
    json.dump(man, open(MANIFEST, 'w'))

    missing = [i['path'] for i in man['items'][:200] if not os.path.exists(i['path'])]
    if missing:
        raise SystemExit(f'{len(missing)}/200 file mẫu không tồn tại, vd {missing[0]} — sai layout dataset')
    print(f"{len(man['items']):,} frame | {len({i['group'] for i in man['items']})} SHOT | "
          f"{len({i['extra'].get('half', '?') for i in man['items']})} hiệp")

    mcfg = ModelConfig(name='siamese', d_model=256, n_layers=6, n_heads=8,
                       img_size=224, patch_size=16, proj_dim=128)
    model = build_model('siamese', mcfg)
    print('tham số:', f"{sum(p.numel() for p in model.parameters()):,}")

    tr = SiamesePairDataset(MANIFEST, 'train', mcfg.img_size)
    va = SiamesePairDataset(MANIFEST, 'val', mcfg.img_size)
    print('cặp train:', len(tr), '| val:', len(va))
    if len(va) == 0:
        # Trainer._evaluate trả None khi loader rỗng → early-stop im lặng vô hiệu, best.pt thành
        # model step cuối mà không có gì báo. Đã gặp thật khi cắt dữ liệu theo tên hiệp.
        raise SystemExit('val rỗng → early-stop sẽ im lặng vô hiệu')

    # Early-stop theo Recall@10 trên VAL. Ở Two-Tower việc này tạo khác biệt thật (best_step rơi vào
    # 1000–3500, không seed nào dính max_steps) nên đừng bỏ ở đây.
    tcfg = TrainConfig(model='siamese', lr=3e-4, weight_decay=0.1, batch_size=128,
                       max_steps=8000, warmup_ratio=0.03, precision='bf16', seed=1337,
                       eval_every=1000, metric_name='recall_at_10',
                       ckpt_dir='/kaggle/working/ckpt_siamese', log_dir='')
    trainer = Trainer(model,
                      DataLoader(tr, batch_size=tcfg.batch_size, shuffle=True, drop_last=True,
                                 num_workers=4, pin_memory=True, persistent_workers=True),
                      DataLoader(va, batch_size=tcfg.batch_size, num_workers=2),
                      tcfg, loss_fn=LOSS_FNS['siamese'], model_cfg=mcfg,
                      compute_metric=_siamese_metric)
    t0 = time.time()
    print(trainer.train())
    print(f'train xong trong {(time.time() - t0) / 60:.0f} phút')

In [ ]:
# T1.3c đã chạy xong ở v22 (Recall@10 0.495, ghi trong docs/). Bật lại khi cần train lại —
# để True thì mỗi lần chạy kernel tốn thêm ~40 phút GPU mà không thêm thông tin gì.
RUN_SIAMESE = False
if RUN_SIAMESE:
    # Eval T1.3c: Recall@1/5/10 + median rank trên 1.000 query (DoD: Recall@10 ≥ 80%),
    # so với 2 baseline evaluation_plan §1 đòi: (a) histogram màu, (b) template matching low-res.
    # Cả ba chấm trên CÙNG 1.000 cặp, cùng một vòng — không hardcode baseline từ run khác
    # (đúng bài học của bug eval T1.7).
    import torch
    from ai_core.baselines import downscale, histogram_features
    from ai_core.scripts.evaluate import recall_at_k

    N_QUERY = 1000
    te = SiamesePairDataset(MANIFEST, 'test', mcfg.img_size)
    idx = torch.randperm(len(te), generator=torch.Generator().manual_seed(0))[:min(N_QUERY, len(te))]
    A = torch.stack([te[int(i)][0] for i in idx])          # query view
    B = torch.stack([te[int(i)][1] for i in idx])          # target view (cùng shot, cách 1–10s)
    print('query:', A.shape[0])

    best = load_checkpoint('/kaggle/working/ckpt_siamese/siamese/best.pt', map_location=trainer.device)
    model.load_state_dict(best['state_dict'])
    model.eval()


    def report(name, ea, eb):
        """ea, eb đã L2-norm → dot = cosine. Rank của cặp đúng (đường chéo) trong toàn bộ target."""
        s = ea @ eb.T
        ranked = s.argsort(dim=1, descending=True).cpu().tolist()
        rel = [{i} for i in range(len(ranked))]
        rank = [r.index(i) + 1 for i, r in enumerate(ranked)]
        med = sorted(rank)[len(rank) // 2]
        print(f'{name:28s} R@1={recall_at_k(ranked, rel, 1):.3f}  R@5={recall_at_k(ranked, rel, 5):.3f}'
              f'  R@10={recall_at_k(ranked, rel, 10):.3f}  median rank={med}')
        return recall_at_k(ranked, rel, 10)


    with torch.no_grad():
        dev = trainer.device
        ea = torch.cat([model.encode(A[i:i + 64].to(dev)).cpu() for i in range(0, len(A), 64)])
        eb = torch.cat([model.encode(B[i:i + 64].to(dev)).cpu() for i in range(0, len(B), 64)])
    r_model = report('Siamese ViT', ea, eb)

    # Baseline (a): histogram màu — chuẩn hoá L2 để cùng thang cosine với model.
    ha = torch.nn.functional.normalize(histogram_features(A), dim=-1)
    hb = torch.nn.functional.normalize(histogram_features(B), dim=-1)
    r_hist = report('baseline histogram màu', ha, hb)

    # Baseline (b): template matching low-res = NN trên pixel hạ 16×16 (DSP cổ điển, không học).
    da = torch.nn.functional.normalize(downscale(A, 16), dim=-1)
    db = torch.nn.functional.normalize(downscale(B, 16), dim=-1)
    r_tpl = report('baseline template 16×16', da, db)

    print()
    print(f"DoD T1.3c (Recall@10 ≥ 0.80): {'PASS' if r_model >= 0.80 else 'FAIL'}")
    print(f"vượt cả 2 baseline: {'PASS' if r_model > max(r_hist, r_tpl) else 'FAIL'}")

## 2b-bis. Run A — shot groups + Siamese (T1.3c) + embedding 1fps (T1.5a)

Chạy trên 80 trận / 160 hiệp / 443k keyframe. Ba bước trong MỘT kernel vì Kaggle chỉ giữ
output của bản mới nhất: tách ra thì checkpoint Siamese bị run sau ghi đè.

In [ ]:
# Run A (shot groups + Siamese + embedding). Tat khi da co embedding va chi train fusion.
RUN_A = False
if RUN_A:
    # Run A: shot groups -> train Siamese (T1.3c) -> trich embedding 1fps cho ca 160 hiep (T1.5a).
    #
    # Gop ba buoc vao MOT kernel vi ca ba dung chung 443k JPEG da upload, va Kaggle chi giu output cua
    # ban MOI NHAT — tach ra thi checkpoint Siamese bi run sau ghi de va mat (da xay ra that: ckpt cua
    # v22 bi v23 xoa, khong con duong nao lay lai).
    #
    # Bay da can that, giu lai de khong lap:
    #  1. Kaggle TU GIAI NEN archive khi tao dataset -> /kaggle/input co the la thu muc, khong co .zip.
    #  2. Manifest sinh tren Windows nen path la 'D:/...'. Phai dung lai ten file tu group + sec.
    #  3. /kaggle/input la filesystem MANG — doc 443k JPEG le tu do rat cham. Copy sang dia local truoc.
    #  4. NHUNG /kaggle/working la thu muc OUTPUT: copy anh vao do thi `kernels output` sau nay tai lai
    #     ca 5,9 GB anh. Dung /tmp.
    import glob, json, os, shutil, sys, time, zipfile, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import SiamesePairDataset
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts import build_shot_groups, extract_vit_embeddings
    from ai_core.scripts.train import _siamese_metric

    JPG_DIR = 'frames_jpg_80'
    LOCAL = '/tmp/frames_local'          # KHONG /kaggle/working: xem bay 4

    # ---------------------------------------------------------------- 1. nap anh + sua path manifest
    # Tim thu muc THAT SU chua cac group, khong tim theo TEN. Kaggle giai nen zip vao mot folder trung
    # ten nen duong dan thanh 'frames_jpg_80/frames_jpg_80/g00h1/...' — glob theo ten se tra ve ca hai
    # tang va sorted() khong bao dam tang nao truoc. Neo vao mot file .jpg that roi lui hai cap.
    sample = glob.glob(f'/kaggle/input/**/{JPG_DIR}/*/*.jpg', recursive=True)
    if sample:
        FRAMES_ROOT = os.path.dirname(os.path.dirname(sample[0]))
    else:
        z = glob.glob('/kaggle/input/**/*.zip', recursive=True)
        if not z:
            raise SystemExit(f'Khong thay {JPG_DIR}/*/*.jpg lan .zip nao trong /kaggle/input')
        with zipfile.ZipFile(z[0]) as f:
            f.extractall('/tmp/frames_unzip')
        inner = glob.glob(f'/tmp/frames_unzip/**/{JPG_DIR}/*/*.jpg', recursive=True)
        if not inner:
            raise SystemExit(f'Giai nen {z[0]} nhung khong thay {JPG_DIR}/*/*.jpg')
        FRAMES_ROOT = os.path.dirname(os.path.dirname(inner[0]))
    print('frames root:', FRAMES_ROOT, f'({len(os.listdir(FRAMES_ROOT))} group)')

    DEST = os.path.join(LOCAL, JPG_DIR)
    if not os.path.isdir(DEST):
        t = time.time()
        shutil.copytree(FRAMES_ROOT, DEST)
        print(f'copy -> dia local trong {(time.time() - t) / 60:.1f} phut', flush=True)

    man_path = glob.glob('/kaggle/input/**/frames_jpg_80.json', recursive=True)
    if not man_path:
        raise SystemExit('Thieu frames_jpg_80.json trong /kaggle/input')
    man = json.load(open(man_path[0], encoding='utf-8'))
    for it in man['items']:
        g, sec = it['group'], it['extra']['sec']
        it['path'] = f'{LOCAL}/{JPG_DIR}/{g}/{g}_s{sec:05d}.jpg'   # dung lai, khong parse path Windows
        it['extra'].pop('src', None)
    FULL_MANIFEST = '/tmp/frames_1fps.json'
    json.dump(man, open(FULL_MANIFEST, 'w'), ensure_ascii=False)

    missing = [i['path'] for i in man['items'][:300] if not os.path.exists(i['path'])]
    if missing:
        raise SystemExit(f'{len(missing)}/300 file mau khong ton tai, vd {missing[0]} — sai layout')
    print(f"{len(man['items']):,} frame | {len({i['group'] for i in man['items']})} hiep", flush=True)

    # ---------------------------------------------------------------- 2. shot groups (dataset_spec 3.2)
    SHOT_MANIFEST = '/tmp/shots_80.json'
    if not os.path.exists(SHOT_MANIFEST):
        sys.argv = ['build_shot_groups', '--manifest', FULL_MANIFEST,
                    '--out-manifest', SHOT_MANIFEST, '--max-frames', '100000']
        t = time.time()
        build_shot_groups.main()
        print(f'shot groups xong trong {(time.time() - t) / 60:.1f} phut', flush=True)

    # ---------------------------------------------------------------- 3. train Siamese (T1.3c)
    SIA_DIR = '/kaggle/working/ckpt_siamese_80'
    mcfg = ModelConfig(name='siamese', d_model=256, n_layers=6, n_heads=8,
                       img_size=224, patch_size=16, proj_dim=128)
    tr = SiamesePairDataset(SHOT_MANIFEST, 'train', mcfg.img_size)
    va = SiamesePairDataset(SHOT_MANIFEST, 'val', mcfg.img_size)
    print('cap train:', len(tr), '| val:', len(va), flush=True)
    if len(va) == 0:
        # Trainer._evaluate tra None khi loader rong -> early-stop im lang vo hieu (da can o T1.3c).
        raise SystemExit('val rong -> early-stop se im lang vo hieu')

    tcfg = TrainConfig(model='siamese', lr=3e-4, weight_decay=0.1, batch_size=128,
                       max_steps=8000, warmup_ratio=0.03, precision='bf16', seed=1337,
                       eval_every=1000, metric_name='recall_at_10', ckpt_dir=SIA_DIR, log_dir='')
    model = build_model('siamese', mcfg)
    print('tham so Siamese:', f'{sum(p.numel() for p in model.parameters()):,}')
    t = time.time()
    Trainer(model,
            DataLoader(tr, batch_size=tcfg.batch_size, shuffle=True, drop_last=True,
                       num_workers=4, pin_memory=True, persistent_workers=True),
            DataLoader(va, batch_size=tcfg.batch_size, num_workers=2),
            tcfg, loss_fn=LOSS_FNS['siamese'], model_cfg=mcfg,
            compute_metric=_siamese_metric).train()
    SIA_CKPT = f'{SIA_DIR}/siamese/best.pt'
    print(f"Siamese: {load_checkpoint(SIA_CKPT)['metrics']} ({(time.time() - t) / 60:.0f} phut)", flush=True)

    # ---------------------------------------------------------------- 3b. cham DoD T1.3c
    # Recall@1/5/10 + median rank tren 1.000 query, so voi 2 baseline evaluation_plan §1 doi:
    # (a) histogram mau, (b) template matching low-res. Ca ba cham tren CUNG 1.000 cap, cung mot vong —
    # khong hardcode baseline tu run khac (bai hoc tu bug eval T1.7).
    from ai_core.baselines import downscale, histogram_features
    from ai_core.scripts.evaluate import recall_at_k

    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.load_state_dict(load_checkpoint(SIA_CKPT, map_location=dev)['state_dict'])
    model.to(dev).eval()

    te = SiamesePairDataset(SHOT_MANIFEST, 'test', mcfg.img_size)
    idx = torch.randperm(len(te), generator=torch.Generator().manual_seed(0))[:min(1000, len(te))]
    A = torch.stack([te[int(i)][0] for i in idx])          # query view
    B = torch.stack([te[int(i)][1] for i in idx])          # target view (cung shot, cach 1-10s)
    print('query:', A.shape[0], flush=True)


    def report(name, ea, eb):
        """ea, eb da L2-norm -> dot = cosine. Rank cua cap dung (duong cheo) trong toan bo target."""
        ranked = (ea @ eb.T).argsort(dim=1, descending=True).cpu().tolist()
        rel = [{i} for i in range(len(ranked))]
        rank = sorted(r.index(i) + 1 for i, r in enumerate(ranked))
        print(f'{name:28s} R@1={recall_at_k(ranked, rel, 1):.3f}  R@5={recall_at_k(ranked, rel, 5):.3f}'
              f'  R@10={recall_at_k(ranked, rel, 10):.3f}  median rank={rank[len(rank) // 2]}')
        return recall_at_k(ranked, rel, 10)


    with torch.no_grad():
        ea = torch.cat([model.encode(A[i:i + 64].to(dev)).float().cpu() for i in range(0, len(A), 64)])
        eb = torch.cat([model.encode(B[i:i + 64].to(dev)).float().cpu() for i in range(0, len(B), 64)])
    r_model = report('Siamese ViT', ea, eb)

    norm = torch.nn.functional.normalize
    r_hist = report('baseline histogram mau', norm(histogram_features(A), dim=-1),
                    norm(histogram_features(B), dim=-1))
    r_tpl = report('baseline template 16x16', norm(downscale(A, 16), dim=-1),
                   norm(downscale(B, 16), dim=-1))
    print()
    print(f"DoD T1.3c (Recall@10 >= 0.80): {'PASS' if r_model >= 0.80 else 'FAIL'}")
    print(f"vuot ca 2 baseline: {'PASS' if r_model > max(r_hist, r_tpl) else 'FAIL'}", flush=True)

    # ---------------------------------------------------------------- 4. embedding 1fps cho MOI hiep
    OUT_EMB = '/kaggle/working/soccernet_vision_80.pt'
    sys.argv = ['extract_vit_embeddings', '--checkpoint', SIA_CKPT, '--manifest', FULL_MANIFEST,
                '--out', OUT_EMB, '--batch-size', '256']
    t = time.time()
    extract_vit_embeddings.main()
    emb = torch.load(OUT_EMB, map_location='cpu', weights_only=False)
    print(f'embedding {len(emb)} hiep trong {(time.time() - t) / 60:.0f} phut | '
          f'{os.path.getsize(OUT_EMB) / 1e6:.0f} MB')

## 2c. Highlight Fusion — cross-attention vision+audio (T1.5)

Feature đã trích sẵn ở local (không upload được video gốc: SoccerNet NDA + 300 GB):
`vision` = embedding backbone Siamese đóng băng (256-d/giây), `audio` = 64-d/giây, `labels` = {0,1,-1}
theo `dataset_spec §3.4`. Ở đây chỉ TRAIN; chấm DoD (sweep τ trên val + so 2 baseline) chạy local
vì baseline cần WAV/PNG gốc, không có trên Kaggle.

In [ ]:
# Run B (sweep fusion). Can highlight_features.pt — chi bat sau khi Run A xong va da ghep feature.
RUN_FUSION = False
if RUN_FUSION:
    # Train HighlightFusion. Kien truc HARDCODE 2 self + 2 cross theo model_spec §2 nen n_layers bi bo
    # qua; num chinh con lai la d_model + dropout.
    #
    # VONG 3 — du lieu tang tu 20 len 80 tran (155 hiep sau khi loai 5 hiep audio hong o nguon).
    # Vong 1 (5,43M tham so, khong dropout, eval_every 250): F1 test 0.1297, THUA baseline RMS 0.1369.
    # Vong 2 (0,37M tham so, dropout 0.1, eval_every 50): 0.1689, vuot ca hai baseline nhung DoD doi
    # +0.10. Ca hai vong deu dat dinh val o step 100-450 roi tut — dau hieu THIEU DU LIEU, khong phai
    # sai kich thuoc model. Nen vong nay: nhieu du lieu hon, va mo lai config lon vi 4x du lieu co the
    # nuoi duoc d_model 256; max_steps 4000 vi mot epoch gio dai gap 4.
    #
    # CHON CONFIG CHI BANG VAL. Tap test khong xuat hien o day va khong bao gio cham tren Kaggle — no
    # chi chay dung mot lan o local qua ai_core.scripts.eval_highlight (evaluation_plan §0.2).
    import glob, time, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import HighlightWindowDataset
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts.train import _highlight_metric

    f = glob.glob('/kaggle/input/**/highlight_features.pt', recursive=True)
    if not f:
        raise SystemExit('Thiếu highlight_features.pt trong /kaggle/input')
    raw = torch.load(f[0], map_location='cpu', weights_only=False)
    print({k: len(v) for k, v in raw.items()})

    WIN = 120
    VDIM, ADIM = raw['train'][0]['vision'].shape[-1], raw['train'][0]['audio'].shape[-1]
    tr = HighlightWindowDataset(raw['train'], WIN, WIN // 2)
    va = HighlightWindowDataset(raw['val'], WIN, WIN // 2)
    print(f'cửa sổ train {len(tr)} | val {len(va)} | vision {VDIM}d audio {ADIM}d')
    if len(va) == 0:
        # Trainer._evaluate trả None khi loader rỗng → early-stop im lặng vô hiệu (đã cắn ở T1.3c).
        raise SystemExit('val rỗng → early-stop sẽ im lặng vô hiệu')

    loaders = (DataLoader(tr, batch_size=32, shuffle=True, drop_last=True, num_workers=2, pin_memory=True),
               DataLoader(va, batch_size=32))


    def run(d_model, dropout, seed, steps, ckpt_dir):
        mcfg = ModelConfig(name='highlight_fusion', d_model=d_model, n_heads=8, dropout=dropout,
                           vision_dim=VDIM, audio_dim=ADIM, window_seconds=WIN)
        tcfg = TrainConfig(model='highlight_fusion', lr=3e-4, weight_decay=0.1, batch_size=32,
                           max_steps=steps, warmup_ratio=0.05, precision='bf16', seed=seed,
                           eval_every=100, metric_name='f1', ckpt_dir=ckpt_dir, log_dir='')
        model = build_model('highlight_fusion', mcfg)
        Trainer(model, *loaders, tcfg, loss_fn=LOSS_FNS['highlight_fusion'],
                model_cfg=mcfg, compute_metric=_highlight_metric).train()
        return load_checkpoint(ckpt_dir + '/highlight_fusion/best.pt')['metrics']


    CONFIGS = [(64, 0.1), (128, 0.1), (128, 0.3), (256, 0.1), (256, 0.3)]
    scores = {}
    for d_model, dropout in CONFIGS:
        t0 = time.time()
        m = run(d_model, dropout, 1337, 4000, f'/kaggle/working/sweep_d{d_model}_p{int(dropout * 10)}')
        scores[(d_model, dropout)] = m['f1']
        print(f"d={d_model:3} dropout={dropout}: val F1 {m['f1']:.4f} @ step {m['step']:4} "
              f"({(time.time() - t0) / 60:.1f} phút)", flush=True)

    best = max(scores, key=scores.get)
    print()
    print(f'CONFIG THANG tren val: d_model={best[0]} dropout={best[1]} (F1 {scores[best]:.4f})')

    for seed in (1337, 1338, 1339):
        m = run(*best, seed, 4000, f'/kaggle/working/ckpt_fusion_best_s{seed}')
        print(f"seed {seed}: val F1 {m['f1']:.4f} @ step {m['step']}", flush=True)

## 2d. Run C — độ nhạy theo thiết kế nhãn (T1.5)

6 biến thể dilation × tập lớp dương, cùng một kiến trúc (d 256, dropout 0.3). Chỉ dùng val;
chọn biến thể bằng khoảng cách model − baseline chấm ở local, không bằng F1 thô.

In [ ]:
# Run C (sweep THIET KE NHAN). Tat khi da chon xong bien the nhan.
RUN_LABEL_SWEEP = False
if RUN_LABEL_SWEEP:
    # Run C: do DO NHAY cua ket qua theo THIET KE NHAN.
    #
    # Vi sao can: dilation va tap lop duong quyet dinh TRAN F1 ma moi model sau do co the dat, va chua
    # bao gio duoc do. Neu tran bi chinh nhan chan thi them du lieu bao nhieu cung khong toi duoc DoD.
    #
    # CANH BAO doc ket qua: F1 giua cac bien the KHONG so truc tiep duoc — bien the 11% duong de dat F1
    # hon han bien the 0,6%. Cell nay chi TRAIN va bao val F1 tho; chon thang thua lam o local bang
    # khoang cach model - baseline tren CUNG mot bien the (eval_highlight --variants).
    #
    # Chi dung config da thang o vong 3 (d_model 256, dropout 0.3) — dang do NHAN, khong do kien truc.
    # Tap test khong xuat hien o day (evaluation_plan §0.2).
    import glob, time, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import HighlightWindowDataset
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts.train import _highlight_metric

    f = glob.glob('/kaggle/input/**/highlight_features.pt', recursive=True)
    v = glob.glob('/kaggle/input/**/label_variants.pt', recursive=True)
    if not f or not v:
        raise SystemExit('Thieu highlight_features.pt hoac label_variants.pt trong /kaggle/input')
    raw = torch.load(f[0], map_location='cpu', weights_only=False)
    variants = torch.load(v[0], map_location='cpu', weights_only=False)
    print({k: len(x) for k, x in raw.items()}, '| bien the:', list(variants))

    WIN = 120
    VDIM, ADIM = raw['train'][0]['vision'].shape[-1], raw['train'][0]['audio'].shape[-1]


    def patched(split, per_stem):
        """Thay mang nhan, giu nguyen vision/audio da tinh. Do dai da duoc can o local."""
        out = []
        for d in raw[split]:
            lab = per_stem[d['name']].long()
            assert lab.shape[0] == d['labels'].shape[0], (d['name'], lab.shape, d['labels'].shape)
            out.append({**d, 'labels': lab})
        return out


    results = {}
    for name, per_stem in variants.items():
        tr = HighlightWindowDataset(patched('train', per_stem), WIN, WIN // 2)
        va = HighlightWindowDataset(patched('val', per_stem), WIN, WIN // 2)
        if len(va) == 0:
            raise SystemExit(f'{name}: val rong -> early-stop im lang vo hieu')

        mcfg = ModelConfig(name='highlight_fusion', d_model=256, n_heads=8, dropout=0.3,
                           vision_dim=VDIM, audio_dim=ADIM, window_seconds=WIN)
        ckpt_dir = f'/kaggle/working/ckpt_lab_{name}'
        tcfg = TrainConfig(model='highlight_fusion', lr=3e-4, weight_decay=0.1, batch_size=32,
                           max_steps=4000, warmup_ratio=0.05, precision='bf16', seed=1337,
                           eval_every=100, metric_name='f1', ckpt_dir=ckpt_dir, log_dir='')
        t0 = time.time()
        Trainer(build_model('highlight_fusion', mcfg),
                DataLoader(tr, batch_size=32, shuffle=True, drop_last=True,
                           num_workers=2, pin_memory=True),
                DataLoader(va, batch_size=32),
                tcfg, loss_fn=LOSS_FNS['highlight_fusion'], model_cfg=mcfg,
                compute_metric=_highlight_metric).train()
        m = load_checkpoint(f'{ckpt_dir}/highlight_fusion/best.pt')['metrics']
        results[name] = m['f1']
        pos = sum(int((t == 1).sum()) for t in per_stem.values())
        tot = sum(int((t != -1).sum()) for t in per_stem.values())
        print(f"{name:8} duong {100 * pos / tot:5.2f}% | val F1 {m['f1']:.4f} @ step {m['step']:4} "
              f"({(time.time() - t0) / 60:.1f} phut)", flush=True)

    print()
    print('val F1 tho (KHONG so truc tiep duoc giua cac bien the):')
    for k, s in sorted(results.items(), key=lambda x: -x[1]):
        print(f'  {k:8} {s:.4f}')

## 2f. Run E — ablation đa phương thức (T1.5e)

`evaluation_plan §2.1` đòi chứng minh đóng góp của đa phương thức: chỉ-visual vs chỉ-audio vs
fusion cross-attention.

**Năm biến thể, không phải ba.** Ba biến thể hiển nhiên bị nhiễu dung lượng: fusion có 5.427.713
tham số còn nhánh đơn theo spec chỉ 1.672.961 — gấp 3,2 lần. Fusion thắng thì chưa phân biệt được
"nhờ đa phương thức" với "nhờ nhiều tham số hơn". Nên thêm 2 đối chứng `n_self=6` đạt 90%/89% dung
lượng của fusion.

Mọi biến thể dùng **cùng một nhãn**, nên val F1 ở đây so trực tiếp được (khác Run C: nhãn khác nhau
thì F1 không so được). Tập test không xuất hiện ở đây; sweep τ trên val chạy ở local qua
`eval_highlight --ablation`.


In [ ]:
# Run E (ablation T1.5e: chi-visual vs chi-audio vs fusion). Tat khi da co bang ablation.
RUN_ABLATION = True
if RUN_ABLATION:
    # Run E — T1.5e (evaluation_plan §2.1): chung minh dong gop DA PHUONG THUC.
    #
    # NAM bien the, khong phai ba. Ba bien the "hien nhien" (fusion/visual/audio) bi NHIEU DUNG LUONG:
    # fusion 5.427.713 tham so vs nhanh don theo spec 1.672.961 — gap 3,2 lan. Fusion thang thi khong
    # ket luan duoc la nho da phuong thuc, vi cung co the chi nho nhieu tham so hon. Hai doi chung
    # n_self=6 (visual_deep 4.886.273, audio_deep 4.837.121 = 90%/89% cua fusion) tra loi dung cau hoi
    # do: fusion van thang doi chung -> khac biet den TU MODALITY.
    #
    # Cung mot NHAN cho moi bien the nen val F1 o day SO TRUC TIEP DUOC (khac Run C: nhan khac nhau
    # thi F1 khong so duoc). Tap test khong xuat hien o day (evaluation_plan §0.2).
    #
    # Config giu dung bien the thang vong 3 (d_model 256, dropout 0.3, 4000 step) — dang do MODALITY,
    # khong do lai kien truc. 5 bien the x 3 seed = 15 run, ~2,5 gio T4.
    import glob, time, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import HighlightWindowDataset
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts.train import _highlight_metric

    # Chi dinh RO ten file: v1 'highlight_features.pt' va v2 'highlight_features_v2.pt' cung
    # nam trong /kaggle/input, glob 'highlight_features*.pt' se khop ca hai va lay file nao
    # la tuy thu tu tra ve — A/B se so hai thu khong biet la thu nao.
    ABL_FEATURES = 'highlight_features_v2.pt'
    f = glob.glob(f'/kaggle/input/**/{ABL_FEATURES}', recursive=True)
    if not f:
        raise SystemExit('Thieu highlight_features.pt trong /kaggle/input')
    raw = torch.load(f[0], map_location='cpu', weights_only=False)
    print({k: len(x) for k, x in raw.items()})

    WIN = 120
    VDIM, ADIM = raw['train'][0]['vision'].shape[-1], raw['train'][0]['audio'].shape[-1]
    tr = HighlightWindowDataset(raw['train'], WIN, WIN // 2)
    va = HighlightWindowDataset(raw['val'], WIN, WIN // 2)
    if len(va) == 0:
        raise SystemExit('val rong -> early-stop im lang vo hieu')
    print('cua so train', len(tr), '| val', len(va))

    VARIANTS = {
        'fusion':      dict(modality='fusion'),                 # spec model_spec §2
        'visual':      dict(modality='visual'),                 # spec: 2 self
        'audio':       dict(modality='audio'),
        'visual_deep': dict(modality='visual', n_self=6),       # doi chung khop dung luong
        'audio_deep':  dict(modality='audio', n_self=6),
        # Cung kien truc fusion, chay tren feature audio v2 (20 mel x mean/max/onset, khong
        # con 28 chieu dem 0). Ten rieng de checkpoint khong ghi de bo cu.
        'fusionv2':    dict(modality='fusion'),
    }
    SEEDS = (1337, 1338, 1339)

    # LAN CHAY NAY: them 4 seed CHI cho fusion. Ly do: voi n=3, fusion (0.2083 +/- 0.0131) va
    # chi-audio (0.1897 +/- 0.0060) co khoang gia tri CHONG NHAU (t=1.83, df=2.8, p~0.17) nen
    # khong ket luan duoc. Phuong sai cua fusion ap dao nen them seed cho fusion la don bay tot
    # nhat: n=7 vs 3 dua t len ~2.5.
    #
    # Day KHONG phai di tim seed dep: n tang cho mot phep so sanh DA DINH TRUOC, va bao cao phai
    # ghi CA 7 seed. Xoa 3 dong duoi de chay lai tron bo 5 bien the x 3 seed.
    ONLY = ('fusionv2',)
    SEEDS = (1337, 1338, 1339, 1340, 1341, 1342, 1343)
    VARIANTS = {k: v for k, v in VARIANTS.items() if k in ONLY}

    results = {}
    for name, kw in VARIANTS.items():
        f1s, n_par = [], 0
        for seed in SEEDS:
            mcfg = ModelConfig(name='highlight_fusion', d_model=256, n_heads=8, dropout=0.3,
                               vision_dim=VDIM, audio_dim=ADIM, window_seconds=WIN, **kw)
            model = build_model('highlight_fusion', mcfg)
            n_par = sum(p.numel() for p in model.parameters())
            ckpt_dir = f'/kaggle/working/ckpt_abl_{name}_s{seed}'
            tcfg = TrainConfig(model='highlight_fusion', lr=3e-4, weight_decay=0.1, batch_size=32,
                               max_steps=4000, warmup_ratio=0.05, precision='bf16', seed=seed,
                               eval_every=100, metric_name='f1', ckpt_dir=ckpt_dir, log_dir='')
            t0 = time.time()
            Trainer(model,
                    DataLoader(tr, batch_size=32, shuffle=True, drop_last=True,
                               num_workers=2, pin_memory=True),
                    DataLoader(va, batch_size=32),
                    tcfg, loss_fn=LOSS_FNS['highlight_fusion'], model_cfg=mcfg,
                    compute_metric=_highlight_metric).train()
            m = load_checkpoint(f'{ckpt_dir}/highlight_fusion/best.pt')['metrics']
            f1s.append(m['f1'])
            print(f"{name:12} s{seed} | {n_par:>9,} tham so | val F1 {m['f1']:.4f} "
                  f"@ step {m['step']:4} ({(time.time() - t0) / 60:.1f} phut)", flush=True)
        mean = sum(f1s) / len(f1s)
        std = (sum((x - mean) ** 2 for x in f1s) / len(f1s)) ** 0.5
        results[name] = (mean, std, n_par)

    print()
    print('val F1 @ tau=0.5 (tau toi uu sweep o local bang eval_highlight --ablation):')
    for k, (mean, std, n_par) in sorted(results.items(), key=lambda x: -x[1][0]):
        print(f'  {k:12} {mean:.4f} +/- {std:.4f}  ({n_par:>9,} tham so)')
    fus = results.get('fusion', (0,))[0]
    for k in ('visual', 'audio', 'visual_deep', 'audio_deep'):
        if k in results:
            print(f'  fusion - {k:12} = {fus - results[k][0]:+.4f}')


## 2e. Run D — ASR CTC trên VIVOS (T1.4c/d)

Split train/test chính thức của VIVOS; val cắt từ train theo NGƯỜI NÓI (5/46).
SpecAugment chỉ bật cho train. Chấm WER/CER trên 760 câu test của 19 người nói khác hẳn.

In [ ]:
# Run D (train ASR CTC + cham WER/CER). Tat khi da co checkpoint asr_ctc.
RUN_ASR = False
if RUN_ASR:
    # Run D — T1.4c/d: train ASR CTC tren VIVOS + cham WER/CER tren VIVOS TEST chinh thuc.
    #
    # Split: giu nguyen train/test cua tac gia VIVOS; val cat tu train THEO NGUOI NOI (5/46 nguoi).
    # Cat val theo cau thi cung mot giong co mat o ca train lan val, early-stop se lac quan vi model
    # da quen giong do — trong khi test la 19 nguoi hoan toan khac.
    #
    # Bay da can that o cac task truoc, giu lai de khong lap:
    #  1. Kaggle TU GIAI NEN zip -> /kaggle/input co the la thu muc, khong con .zip.
    #  2. Manifest sinh tren Windows: dung lai path tu group + ten file, khong parse path cu.
    #  3. /kaggle/input la filesystem MANG -> copy sang dia local truoc khi train nhieu epoch.
    #  4. /kaggle/working la thu muc OUTPUT -> copy wav vao do thi `kernels output` tai lai ca 1,8 GB.
    import glob, json, os, shutil, time, zipfile, torch
    from torch.utils.data import DataLoader
    from ai_core import build_model
    from ai_core.config import ModelConfig, TrainConfig
    from ai_core.training import Trainer
    from ai_core.training.checkpoint import load_checkpoint
    from ai_core.training.datasets import ASRDataset, CharVocab, collate_ctc
    from ai_core.training.loss_fns import LOSS_FNS
    from ai_core.scripts.train import _asr_metric

    WAVES, LOCAL = 'vivos_waves', '/tmp/vivos_local'

    sample = glob.glob(f'/kaggle/input/**/{WAVES}/*/*.wav', recursive=True)
    if sample:
        ROOT = os.path.dirname(os.path.dirname(sample[0]))
    else:
        z = glob.glob('/kaggle/input/**/vivos_waves.zip', recursive=True)
        if not z:
            raise SystemExit('Khong thay vivos_waves/*/*.wav lan vivos_waves.zip trong /kaggle/input')
        with zipfile.ZipFile(z[0]) as f:
            f.extractall('/tmp/vivos_unzip')
        inner = glob.glob(f'/tmp/vivos_unzip/**/{WAVES}/*/*.wav', recursive=True)
        ROOT = os.path.dirname(os.path.dirname(inner[0]))
    print('waves root:', ROOT, f'({len(os.listdir(ROOT))} nguoi noi)')

    DEST = os.path.join(LOCAL, WAVES)
    if not os.path.isdir(DEST):
        t = time.time()
        shutil.copytree(ROOT, DEST)
        print(f'copy -> dia local trong {(time.time() - t) / 60:.1f} phut', flush=True)

    man_path = glob.glob('/kaggle/input/**/vivos.json', recursive=True)
    voc_path = glob.glob('/kaggle/input/**/vivos_vocab.json', recursive=True)
    if not man_path or not voc_path:
        raise SystemExit('Thieu vivos.json hoac vivos_vocab.json trong /kaggle/input')
    man = json.load(open(man_path[0], encoding='utf-8'))
    for it in man['items']:
        it['path'] = f"{DEST}/{it['group']}/{os.path.basename(it['path'])}"
    MANIFEST = '/tmp/vivos_kaggle.json'
    json.dump(man, open(MANIFEST, 'w'), ensure_ascii=False)

    missing = [i['path'] for i in man['items'][:300] if not os.path.exists(i['path'])]
    if missing:
        raise SystemExit(f'{len(missing)}/300 file mau khong ton tai, vd {missing[0]}')

    raw_vocab = json.load(open(voc_path[0], encoding='utf-8'))
    chars = raw_vocab.get('chars', raw_vocab) if isinstance(raw_vocab, dict) else raw_vocab
    vocab = CharVocab(chars)
    print(f"{len(man['items']):,} cau | vocab CTC {vocab.size} (blank + {len(chars)} ky tu)")

    mcfg = ModelConfig(name='asr_ctc', d_model=256, n_layers=8, n_heads=8, dropout=0.1,
                       n_mels=80, n_chars=vocab.size)
    # augment=True CHI cho train. SpecAugment la nn.Module, thoat som khi self.training False —
    # ASRDataset goi .train() cho no. Bat cho val thi so do dao dong theo may rui tung lan chay.
    tr = ASRDataset(MANIFEST, 'train', vocab, n_mels=mcfg.n_mels, augment=True)
    va = ASRDataset(MANIFEST, 'val', vocab, n_mels=mcfg.n_mels)
    te = ASRDataset(MANIFEST, 'test', vocab, n_mels=mcfg.n_mels)
    print('cau train:', len(tr), '| val:', len(va), '| test:', len(te))
    if len(va) == 0:
        raise SystemExit('val rong -> early-stop im lang vo hieu')

    tcfg = TrainConfig(model='asr_ctc', lr=3e-4, weight_decay=0.01, batch_size=16, max_steps=20000,
                       warmup_ratio=0.05, precision='bf16', seed=1337, eval_every=2000,
                       metric_name='wer', ckpt_dir='/kaggle/working/ckpt_asr', log_dir='')
    model = build_model('asr_ctc', mcfg)
    print('tham so:', f'{sum(p.numel() for p in model.parameters()):,}')

    t0 = time.time()
    Trainer(model,
            DataLoader(tr, batch_size=tcfg.batch_size, shuffle=True, drop_last=True,
                       num_workers=4, pin_memory=True, persistent_workers=True, collate_fn=collate_ctc),
            DataLoader(va, batch_size=tcfg.batch_size, num_workers=2, collate_fn=collate_ctc),
            tcfg, loss_fn=LOSS_FNS['asr_ctc'], model_cfg=mcfg,
            metadata={'char_vocab': vocab.itos},
            compute_metric=_asr_metric(vocab)).train()
    CKPT = f'{tcfg.ckpt_dir}/asr_ctc/best.pt'
    print(f"train xong {(time.time() - t0) / 60:.0f} phut | best: {load_checkpoint(CKPT)['metrics']}",
          flush=True)

    # ---------------------------------------------------------------- T1.4d: WER/CER tren VIVOS TEST
    from ai_core.models.ctc import ctc_greedy_decode
    from ai_core.scripts.evaluate import wer


    def cer(ref, hyp):
        """Levenshtein muc KY TU / do dai ref. Tinh tay vi `wer` cua du an chi lam muc TU."""
        a, b = list(ref), list(hyp)
        prev = list(range(len(b) + 1))
        for i, ca in enumerate(a, 1):
            cur = [i] + [0] * len(b)
            for j, cb in enumerate(b, 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
            prev = cur
        return prev[-1] / max(1, len(a))


    dev = next(model.parameters()).device
    model.load_state_dict(load_checkpoint(CKPT, map_location=str(dev))['state_dict'])
    model.eval()
    wers, cers = [], []
    with torch.no_grad():
        for mel, mel_len, tgt, tgt_len in DataLoader(te, batch_size=16, collate_fn=collate_ctc):
            lp, out_len = model(mel.to(dev), mel_len.to(dev))
            pred = ctc_greedy_decode(lp.float().cpu(), out_len.cpu())
            for i, ids in enumerate(pred):
                ref = vocab.decode(tgt[i, :tgt_len[i]].tolist())
                hyp = vocab.decode(ids)
                wers.append(wer(ref, hyp))
                cers.append(cer(ref, hyp))
    W, C = sum(wers) / len(wers), sum(cers) / len(cers)
    print(f'\nVIVOS test ({len(wers)} cau): WER {W:.4f} | CER {C:.4f}')
    print(f"DoD T1.4d (CER <= 0.20 va WER <= 0.45): {'PASS' if C <= 0.20 and W <= 0.45 else 'FAIL'}")

## 3. Template 5 model còn lại (attach dataset rồi chạy)

Mỗi model cần dataset thật. Attach qua **Add Data**, sửa đường dẫn `TODO`, rồi chạy `scripts.train`.
GPU T4: ViT/Siamese/Fusion vài giờ; ASR/Mini-GPT nặng — chia phiên (Kaggle 30h/tuần, mỗi phiên ~12h).


In [ ]:
# --- ViT phân loại (cần keyframe + nhãn) ---
# 1) Attach dataset ảnh keyframe, tạo manifest bằng scripts.prepare_keyframes
# 2) Train:
# !python -m ai_core.scripts.train \
#     --config /kaggle/input/ai-core/ai_core/training/configs/vit.yaml \
#     --manifest /kaggle/working/manifests/vit.json   # TODO đường dẫn manifest thật
print('ViT: attach keyframe dataset + manifest rồi bỏ comment lệnh train')


In [ ]:
# --- ASR CTC (VIVOS / Common Voice vi) ---
# Attach dataset VIVOS trên Kaggle, dựng CharVocab từ transcript, train:
# !python -m ai_core.scripts.train --config .../asr_ctc.yaml --manifest .../asr.json --vocab .../char_vocab.json
# LƯU Ý: xuất kèm checkpoints/asr_ctc/vocab.json cho inference real path
print('ASR: attach VIVOS, tạo manifest + char_vocab.json')


In [ ]:
# --- Mini-GPT (BPE + pretrain text + fine-tune tóm tắt) ---
# 1) train BPE: !python -m ai_core.scripts.train_bpe --corpus '.../*.txt' --out /kaggle/working/bpe.json
# 2) pretrain: !python -m ai_core.scripts.train --config .../minigpt_pretrain.yaml --manifest ...
# 3) fine-tune tóm tắt: config minigpt_finetune_summary.yaml
print('Mini-GPT: train BPE → pretrain → fine-tune (chạy đêm, chia phiên)')


## 4. Đóng gói checkpoint để tải về

In [ ]:
# Nén checkpoints (best.pt + sidecar) để Download từ panel Output
import shutil, os
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
shutil.make_archive('/kaggle/working/checkpoints_export', 'zip', '/kaggle/working/checkpoints')
print('Tải: /kaggle/working/checkpoints_export.zip')
for r,_,fs in os.walk('/kaggle/working/checkpoints'):
    for f in fs: print(os.path.join(r,f))
